# Gradient Boosting from scratch — NumPy only, checked against scikit-learn

This notebook builds gradient boosting **from nothing but NumPy**: the regression tree, the losses, the boosting loop,
binary and multi-class classification, subsampling and early stopping. Every piece is checked against the README's
hand-worked examples and, at the end, against scikit-learn's `GradientBoostingRegressor` / `GradientBoostingClassifier`
on the same data — **to floating-point precision**.

**How to use it:** run the imports cell first; then top-to-bottom always works (later sections reuse functions defined
earlier). Read the markdown before each code cell — it restates the step from the README. The boxes titled
**Implementation hurdle** show a problem you only meet when you actually write the code, and demonstrate it.
scikit-learn is used only for datasets, metrics, `train_test_split` and the library models we compare with.
The data comes from `data/good_fit.csv` and `data/bad_fit.csv`, written by `gradient_boosting_lab.ipynb` (run that
notebook first if the files are missing).

### Theory → code map

| README section / equation | Function | Notebook section |
|---|---|---|
| §2.3 a split, its squared error; §4.7 Friedman's improvement | `score_splits`, `best_threshold_naive` | 1.1 |
| §2.3 the regression tree (fit to pseudo-residuals) | `fit_regression_tree`, `find_best_split`, `apply_tree`, `predict_tree` | 1.2 |
| §4.1 F₀ = argmin Σ L(y, γ): mean / median | `initial_prediction` | 2 |
| §4.2 pseudo-residual r = −∂L/∂F | `negative_gradient` | 2 |
| §4.3 per-leaf line search γⱼ = argmin Σ L(y, F + γ) | `leaf_value` | 2 |
| §3, §4.5 F_m = F_{m−1} + η·γ (shrink after solving) | `fit_gbm_regressor`, `predict_gbm_regressor` | 3 |
| §4.4 binary: F₀ = log-odds, r = y − p, γ = Σr / Σp(1−p) | `initial_raw_scores`, `binary_negative_gradient`, `newton_leaf_value` | 4 |
| §4.6 K classes: softmax, K trees per round, (K−1)/K | `softmax`, `fit_gbm_classifier` | 5 |
| §6.2 `subsample`: new rows each round, without replacement | `subsample_rows` | 6 |
| §6.2 `n_iter_no_change`: early stopping on a validation split | inside `fit_gbm_regressor` / `fit_gbm_classifier` | 7 |
| §8.6 histogram binning (pointer to `../06-xgboost/`) | `best_threshold_histogram` | 8 |
| — comparison with scikit-learn on both CSVs | — | 9 |

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

DATA = Path("data")
good = np.loadtxt(DATA / "good_fit.csv", delimiter=",", skiprows=1)
bad = np.loadtxt(DATA / "bad_fit.csv", delimiter=",", skiprows=1)
X_good, y_good = good[:, :-1], good[:, -1]
X_bad, y_bad = bad[:, :-1], bad[:, -1]
print("good_fit:", X_good.shape, " bad_fit:", X_bad.shape)

good_fit: (2000, 10)  bad_fit: (480, 2)


---
## 1. The regression tree that every round fits

Every boosting round fits a small **regression tree** to the pseudo-residuals r (README §2.3): it keeps cutting the
rows into two groups, "feature ≤ threshold" and "feature > threshold", choosing the cut that makes the squared error
around each group's mean as small as possible. Each leaf then predicts the mean r of its rows.

### 1.1 Scoring one split, and why Friedman's criterion is the same thing

For a cut that puts n_L rows (sum of residuals S_L) on the left and n_R rows (S_R) on the right, the squared error
left after the cut is Σr² − S_L²/n_L − S_R²/n_R. Σr² does not depend on the cut, so the best cut **maximises**
S_L²/n_L + S_R²/n_R. Friedman (2001, eq. 35) scores a cut by n_L·n_R/(n_L+n_R)·(r̄_L − r̄_R)²; a line of algebra shows
this is exactly the drop in squared error, S_L²/n_L + S_R²/n_R − (S_L+S_R)²/n. That answers the old question
*"why `friedman_mse`?"*: for unweighted data it ranks splits exactly like plain MSE.

Two ways to compute the scores: the **naive** way recomputes both group means for every threshold (n thresholds × n
rows = O(n²) work); the **sorted** way sorts once and gets every S_L from one running (cumulative) sum, O(n log n).

In [2]:
def best_threshold_naive(x, r):
    """Try every threshold between two sorted x values; recompute both groups from scratch each time. O(n^2)."""
    xs = np.unique(x)
    best = (np.inf, None)
    for a, b in zip(xs[:-1], xs[1:]):
        t = (a + b) / 2
        left, right = r[x <= t], r[x > t]
        sse = np.sum((left - left.mean()) ** 2) + np.sum((right - right.mean()) ** 2)
        if sse < best[0]:
            best = (sse, t)
    return best[1], best[0]


def score_splits(x_sorted, r_sorted):
    """S_L^2/n_L + S_R^2/n_R for every cut between different sorted x values, from ONE cumulative sum."""
    n = len(x_sorted)
    positions = np.nonzero(x_sorted[1:] > x_sorted[:-1])[0] + 1    # cut before position p
    s_left = np.cumsum(r_sorted)[positions - 1]
    s_right = r_sorted.sum() - s_left
    return positions, s_left**2 / positions + s_right**2 / (n - positions)


rng = np.random.default_rng(0)
x_demo = rng.uniform(0, 1, 300)
r_demo = np.sin(6 * x_demo) + rng.normal(0, 0.3, 300)
order = np.argsort(x_demo)
xs, rs = x_demo[order], r_demo[order]
positions, scores = score_splits(xs, rs)
p = positions[np.argmax(scores)]
n, n_l = len(rs), positions
friedman = n_l * (n - n_l) / n * (np.cumsum(rs)[n_l - 1] / n_l - (rs.sum() - np.cumsum(rs)[n_l - 1]) / (n - n_l)) ** 2
sse_drop = scores - rs.sum() ** 2 / n
print("sorted + cumsum threshold:", (xs[p - 1] + xs[p]) / 2, "| naive threshold:", best_threshold_naive(x_demo, r_demo)[0])
print("Friedman's improvement == drop in squared error for every cut:", np.allclose(friedman, sse_drop),
      "| same best cut:", np.argmax(friedman) == np.argmax(scores))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    GradientBoostingRegressor(n_estimators=1, criterion="friedman_mse").fit(x_demo.reshape(-1, 1), r_demo)
if caught:   # scikit-learn >= 1.9 deprecates the parameter; older versions accept it silently
    print("scikit-learn when you pass criterion='friedman_mse':", caught[0].category.__name__, "-", str(caught[0].message)[:95])
else:
    print("no warning: this scikit-learn version still uses the criterion parameter")

sorted + cumsum threshold: 0.5180967253284989 | naive threshold: 0.5180967253284989
Friedman's improvement == drop in squared error for every cut: True | same best cut: True
scikit-learn when you pass criterion='friedman_mse': FutureWarning - The parameter `criterion` is deprecated and will be removed in 1.11. It has no effect. Leave it


> **Implementation hurdle — the cost of the split search.** The tree tries every threshold of every feature in
> every node, in every round. The naive version is quadratic in the number of rows; the sorted + cumulative-sum
> version is (almost) linear. Timed below on one feature.

In [3]:
for n_rows in [250, 500, 1000, 2000]:
    x_t, r_t = rng.uniform(0, 1, n_rows), rng.normal(0, 1, n_rows)
    t0 = time.perf_counter(); best_threshold_naive(x_t, r_t); t_naive = time.perf_counter() - t0
    t0 = time.perf_counter()
    o = np.argsort(x_t); score_splits(x_t[o], r_t[o])
    t_fast = time.perf_counter() - t0
    print(f"n = {n_rows:>4}: naive {t_naive * 1e3:8.1f} ms   sorted + cumsum {t_fast * 1e3:6.2f} ms   ({t_naive / t_fast:,.0f}x faster)")

n =  250: naive      1.9 ms   sorted + cumsum   0.06 ms   (30x faster)
n =  500: naive      4.0 ms   sorted + cumsum   0.09 ms   (46x faster)
n = 1000: naive      9.6 ms   sorted + cumsum   0.06 ms   (149x faster)
n = 2000: naive     26.6 ms   sorted + cumsum   0.13 ms   (211x faster)


> **How to read this**
> * Both methods pick the same threshold, and Friedman's formula equals the squared-error drop for every cut, so they
>   always choose the same split. scikit-learn 1.9 has in fact **deprecated** the `criterion` parameter of gradient
>   boosting (it has no effect; trees always use squared error), which is the practical answer to "why friedman_mse?".
> * The naive time grows much faster than n (between ×2 and ×3 per doubling here, heading towards ×4 for larger n),
>   the sorted one stays tiny: at n = 2000 the naive search is a couple of hundred times slower (exact factors vary
>   from run to run) — and that is for one feature in one node of one tree.

### 1.2 The full tree, faithful to scikit-learn

The tree below grows depth-first: at each node it searches all features (§1.1), cuts, and recurses until
`max_depth`, too few rows, or a node whose residuals are all equal. A tree is stored as **flat arrays**
(`feature`, `threshold`, `left`, `right`, `value`) — node *i*'s children are `left[i]` and `right[i]`, and `feature[i] = -1`
marks a leaf. (The idea of keeping an index array and partitioning it per node is the same as in this repo's
`src/fromScratch/decisionTrees/` notebook.)

To match scikit-learn **to the last bit** we copy three of its details; each is an implementation hurdle shown right
after the code: (a) it rounds X to float32, (b) it tries features in a random order and keeps the *first* best one
when two are tied, (c) it adds residuals one by one in a particular order, and with ties that order decides.

In [4]:
FEATURE_THRESHOLD = 1e-7          # x values closer than this count as equal (as in scikit-learn)
EPSILON = np.finfo(np.float64).eps
RAND_R_MAX = 2147483647


def round_to_float32(X):
    """scikit-learn stores X as float32; do the same rounding, then compute in float64."""
    return np.asarray(X, dtype=np.float32).astype(np.float64)


def sequential_sum(values, start=0.0):
    """Add values one by one, left to right (np.sum adds pairwise, which rounds differently)."""
    return np.cumsum(np.concatenate([[start], values]))[-1]


def draw_feature_index(state, low, high):
    """scikit-learn's xorshift generator decides the order in which a node tries the features.
    state['seed'] = None gives a fixed order instead (used only to demonstrate the tie hurdle)."""
    if state["seed"] is None:
        return low
    s = state["seed"] or 1
    s ^= (s << 13) & 0xFFFFFFFF
    s ^= s >> 17
    s ^= (s << 5) & 0xFFFFFFFF
    state["seed"] = s
    return low + (s % (RAND_R_MAX + 1)) % (high - low)


def left_sums(r_sorted, positions, total):
    """Sum of r_sorted[:p] for every candidate cut p, accumulated in scikit-learn's order."""
    n = len(r_sorted)
    previous = np.concatenate([[0], positions[:-1]])
    if np.all(positions - previous <= n - positions):          # usual case: walking forward = a cumulative sum
        return np.cumsum(r_sorted)[positions - 1]
    out, s, pos = np.empty(len(positions)), 0.0, 0
    for j, p in enumerate(positions):                          # rare: a big jump over tied x values
        s = sequential_sum(r_sorted[pos:p], s) if p - pos <= n - p else sequential_sum(-r_sorted[p:][::-1], total)
        out[j], pos = s, p
    return out


def score_node_splits(x_sorted, r_sorted, total, min_samples_leaf):
    """Like score_splits (1.1), plus the minimum leaf size and scikit-learn's summation order."""
    n = len(x_sorted)
    positions = np.nonzero(x_sorted[1:] > x_sorted[:-1] + FEATURE_THRESHOLD)[0] + 1
    positions = positions[(positions >= min_samples_leaf) & (n - positions >= min_samples_leaf)]
    if len(positions) == 0:
        return positions, np.array([])
    s_left = left_sums(r_sorted, positions, total)
    s_right = total - s_left
    return positions, s_left * s_left / positions + s_right * s_right / (n - positions)


def find_best_split(X, r, samples, start, end, total, state, n_known_constant, min_samples_leaf):
    """Best (feature, threshold) for the rows samples[start:end]; features tried in scikit-learn's random order."""
    features, constant = state["features"], state["constant"]
    n_features = len(features)
    f_i, n_visited, n_found, n_drawn, n_constant = n_features, 0, 0, 0, n_known_constant
    best_score, best_feature, best_threshold = -np.inf, -1, np.nan
    while f_i > n_constant and (n_visited < n_features or n_visited <= n_found + n_drawn):
        n_visited += 1
        f_j = draw_feature_index(state, n_drawn, f_i - n_found)
        if f_j < n_known_constant:                             # constant in an ancestor node: skip it
            features[n_drawn], features[f_j] = features[f_j], features[n_drawn]
            n_drawn += 1
            continue
        f_j += n_found
        f = features[f_j]
        seg = samples[start:end]
        seg = seg[np.argsort(X[seg, f], kind="stable")]       # sort this node's rows by feature f
        samples[start:end] = seg
        x_sorted = X[seg, f]
        if x_sorted[-1] <= x_sorted[0] + FEATURE_THRESHOLD:    # constant here: remember it for the children
            features[f_j], features[n_constant] = features[n_constant], features[f_j]
            n_found += 1
            n_constant += 1
            continue
        f_i -= 1
        features[f_i], features[f_j] = features[f_j], features[f_i]
        positions, scores = score_node_splits(x_sorted, r[seg], total, min_samples_leaf)
        if len(scores) == 0:
            continue
        j = int(np.argmax(scores))                             # first best cut of this feature
        if scores[j] > best_score:                             # must be STRICTLY better than earlier features
            p = positions[j]
            threshold = x_sorted[p - 1] / 2.0 + x_sorted[p] / 2.0
            best_score, best_feature = scores[j], f
            best_threshold = x_sorted[p - 1] if threshold == x_sorted[p] else threshold
    features[:n_known_constant] = constant[:n_known_constant]
    constant[n_known_constant:n_known_constant + n_found] = features[n_known_constant:n_known_constant + n_found]
    return best_feature, best_threshold, n_constant


def partition_rows(seg, goes_left):
    """Move left-going rows to the front with scikit-learn's swap-from-the-end scheme (fixes the row order)."""
    seg, goes_left = list(seg), list(goes_left)
    p, end = 0, len(seg)
    while p < end:
        if goes_left[p]:
            p += 1
        else:
            end -= 1
            seg[p], seg[end] = seg[end], seg[p]
            goes_left[p], goes_left[end] = goes_left[end], goes_left[p]
    return np.array(seg, dtype=int), p


def fit_regression_tree(X, r, rows, max_depth, seed, min_samples_split=2, min_samples_leaf=1):
    """Grow a least-squares regression tree on targets r using only `rows`; return it as flat arrays."""
    samples = np.array(rows, dtype=int)
    state = {"seed": seed, "features": np.arange(X.shape[1]), "constant": np.zeros(X.shape[1], dtype=int)}
    tree = {"feature": [], "threshold": [], "left": [], "right": [], "value": []}

    def grow(start, end, depth, impurity, n_known_constant):
        seg, n = samples[start:end], end - start
        total, sq_total = sequential_sum(r[seg]), sequential_sum(r[seg] * r[seg])
        if impurity is None:
            impurity = sq_total / n - (total / n) ** 2          # variance of r in this node
        node = len(tree["feature"])
        for key, val in (("feature", -1), ("threshold", np.nan), ("left", -1), ("right", -1), ("value", total / n)):
            tree[key].append(val)
        if depth >= max_depth or n < min_samples_split or n < 2 * min_samples_leaf or impurity <= EPSILON:
            return node                                         # leaf: predicts the mean of r
        f, threshold, n_constant = find_best_split(X, r, samples, start, end, total, state,
                                                   n_known_constant, min_samples_leaf)
        if f < 0:
            return node
        seg, n_left = partition_rows(samples[start:end], X[samples[start:end], f] <= threshold)
        samples[start:end] = seg
        r_seg = r[seg]
        s_left = (sequential_sum(r_seg[:n_left]) if n_left <= n - n_left
                  else sequential_sum(-r_seg[n_left:][::-1], total))
        sq_left = sequential_sum(r_seg[:n_left] * r_seg[:n_left])
        s_right, sq_right = total - s_left, sq_total - sq_left
        tree["feature"][node], tree["threshold"][node] = f, threshold
        tree["left"][node] = grow(start, start + n_left, depth + 1,
                                  sq_left / n_left - (s_left / n_left) ** 2, n_constant)
        tree["right"][node] = grow(start + n_left, end, depth + 1,
                                   sq_right / (n - n_left) - (s_right / (n - n_left)) ** 2, n_constant)
        return node

    grow(0, len(samples), 0, None, 0)
    return {key: np.array(val) for key, val in tree.items()}


def apply_tree(tree, X):
    """Index of the leaf that every row of X lands in."""
    node = np.zeros(len(X), dtype=int)
    while True:
        inner = np.nonzero(tree["feature"][node] >= 0)[0]
        if len(inner) == 0:
            return node
        nd = node[inner]
        goes_left = X[inner, tree["feature"][nd]] <= tree["threshold"][nd]
        node[inner] = np.where(goes_left, tree["left"][nd], tree["right"][nd])


def predict_tree(tree, X):
    return tree["value"][apply_tree(tree, X)]


# check: one tree on the good-fit residuals vs scikit-learn's DecisionTreeRegressor (same seed)
r_good = y_good - y_good.mean()
seed0 = np.random.RandomState(0).randint(0, RAND_R_MAX)           # the seed scikit-learn derives from random_state=0
own_tree = fit_regression_tree(round_to_float32(X_good), r_good, np.arange(len(r_good)), max_depth=4, seed=seed0)
sk_tree = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X_good, r_good)
X_new = np.random.default_rng(1).uniform(0, 1, (100_000, 10))
print("nodes:", len(own_tree["feature"]), "vs", sk_tree.tree_.node_count,
      "| same features:", np.array_equal(own_tree["feature"], np.where(sk_tree.tree_.feature < 0, -1, sk_tree.tree_.feature)))
print("max |difference| on the 2000 training rows:", np.abs(predict_tree(own_tree, round_to_float32(X_good)) - sk_tree.predict(X_good)).max())
print("max |difference| on 100,000 new rows      :", np.abs(predict_tree(own_tree, round_to_float32(X_new)) - sk_tree.predict(X_new)).max())

nodes: 31 vs 31 | same features: True
max |difference| on the 2000 training rows: 0.0
max |difference| on 100,000 new rows      : 0.0


> **Implementation hurdle (a) — scikit-learn rounds X to float32.** float32 keeps only about 7 significant digits, so
> 1000.0 and 1000.00002 become *the same number* and scikit-learn cannot cut between them. A float64 tree can.

In [5]:
x_close = np.array([[1000.0], [1000.00002], [1001.0], [1002.0]])
r_close = np.array([0.0, 10.0, 10.0, 10.0])
for name, X_in in [("float64 inputs", x_close), ("float32-rounded inputs", round_to_float32(x_close))]:
    t = fit_regression_tree(X_in, r_close, np.arange(4), max_depth=1, seed=seed0)
    print(f"{name:<24}: threshold {t['threshold'][0]:.5f}, leaf values {t['value'][1:]}")
print("scikit-learn             : threshold", f"{DecisionTreeRegressor(max_depth=1).fit(x_close, r_close).tree_.threshold[0]:.5f}")

float64 inputs          : threshold 1000.00001, leaf values [ 0. 10.]
float32-rounded inputs  : threshold 1000.50000, leaf values [ 5. 10.]
scikit-learn             : threshold 1000.50000


> **Implementation hurdle (b) and (c) — ties.** Sometimes two features cut the rows into *exactly the same two
> groups* (common with small nodes, or with residuals that take only a few values — e.g. ±1 for the absolute loss,
> or y − p in the first classification round). Mathematically the scores are equal; which feature wins then depends on
> the order the features are tried and on the last bit of a floating-point sum. Below: (b) the placement data from
> README §5.2 has such a tie at the root's right child; (c) the same eight numbers added in two orders differ in the
> last bit; and an absolute-loss model built with a *fixed* feature order instead of scikit-learn's random order
> drifts away from scikit-learn.

In [6]:
X_place = np.array([[6.82, 118], [6.36, 125], [5.39, 99], [5.50, 106],
                    [6.39, 148], [9.13, 148], [7.17, 147], [7.72, 72]])
y_place = np.array([0, 1, 1, 1, 0, 1, 1, 0])
rows = np.array([0, 4, 5, 6, 7])                                  # the 5 rows with cgpa > 6.375
r_place = y_place - 0.625
for f, name in enumerate(["cgpa", "iq"]):
    o = rows[np.argsort(X_place[rows, f])]
    pos, sc = score_splits(X_place[o, f], r_place[o])
    print(f"best cut on {name:<4}: score {sc.max():.12f}, left rows {o[:pos[np.argmax(sc)]] + 1}")

v = np.array([0.1, 0.2, 0.3, -0.6, 1e-3, 7e5, -7e5, 0.05])
print("\nsame 8 numbers, two summation orders:", repr(sequential_sum(v)), "vs", repr(sequential_sum(v[::-1])))

print()
sk_abs = GradientBoostingRegressor(loss="absolute_error", n_estimators=100, random_state=0).fit(X_good, y_good)
for label, fixed_order in [("scikit-learn's random feature order", False), ("fixed feature order", True)]:
    Xr, F = round_to_float32(X_good), np.full(len(y_good), np.median(y_good))
    rng_tree = np.random.RandomState(0)
    for m in range(100):
        r = np.where(F > y_good, -1.0, 1.0)                     # absolute-loss residuals: only +1 / -1
        seed = rng_tree.randint(0, RAND_R_MAX)
        tree = fit_regression_tree(Xr, r, np.arange(len(r)), 3, None if fixed_order else seed)
        leaves = apply_tree(tree, Xr)
        for leaf in np.unique(leaves):
            tree["value"][leaf] = np.percentile(y_good[leaves == leaf] - F[leaves == leaf], 50, method="inverted_cdf")
        F = F + 0.1 * tree["value"][leaves]
    print(f"absolute loss, 100 rounds, {label:<36}: max |F - scikit-learn| = {np.abs(F - sk_abs.predict(X_good)).max():.2e}")

best cut on cgpa: score 0.786458333333, left rows [5 1]
best cut on iq  : score 0.786458333333, left rows [8 1]

same 8 numbers, two summation orders: np.float64(0.051000000047497454) vs np.float64(0.05100000004656616)



absolute loss, 100 rounds, scikit-learn's random feature order : max |F - scikit-learn| = 2.84e-14


absolute loss, 100 rounds, fixed feature order                 : max |F - scikit-learn| = 3.19e-01


> **How to read this**
> * Float32 hurdle: with float64 inputs our tree cuts between 1000.0 and 1000.00002 (threshold 1000.00001, leaves
>   0 and 10 — a perfect fit); after float32 rounding those two rows are the same number, so the cut has to move to
>   1000.5 (leaves 5 and 10) — exactly where scikit-learn puts it.
> * Placement data: "cgpa ≤ 6.995" puts rows {1, 5} on the left, "iq ≤ 132.5" puts rows {1, 8} there — different rows,
>   but all four are "not placed" with the same residual −0.625, so the **scores are identical to 12 decimals**
>   (0.786458333333). Only the order of trying features decides (Lab 2 of the library notebook shows `random_state`
>   flipping it).
> * The same eight numbers summed in two orders differ in the 10th digit — enough to break a tie the other way.
> * With scikit-learn's feature order (and summation order) our absolute-loss model is identical to scikit-learn after
>   100 rounds (difference 3e-14); with a fixed order it has drifted by 0.32 — the ±1 residuals of the absolute loss
>   create many exact ties. Neither model is "better"; they just break ties differently.

---
## 2. Losses: the starting value F₀, the pseudo-residual, the leaf line search

README §4.1–4.3. For each loss we need three things:

| Loss L(y, F) | F₀ = argmin_γ Σ L(y, γ) | pseudo-residual r = −∂L/∂F | leaf value γⱼ = argmin_γ Σ_{leaf} L(y, F + γ) |
|---|---|---|---|
| squared ½(y − F)² | mean of y | y − F | mean of r in the leaf (the tree already stores it) |
| absolute \|y − F\| | median of y | sign(y − F) | median of (y − F) in the leaf |
| Huber (δ) | median of y | y − F, capped at ±δ | median + mean of capped deviations (one step) |

For Huber, scikit-learn re-sets δ every round to the `alpha`-quantile (default 0.9) of the current |y − F|.

In [7]:
def initial_prediction(y, loss):
    """F0 = the best constant: mean for squared error, median for absolute and Huber."""
    return np.mean(y) if loss == "squared_error" else np.median(y)


def huber_delta(y, F, alpha):
    """Huber's switch point: the alpha-quantile of the current absolute errors."""
    return np.percentile(np.abs(y - F), 100 * alpha, method="inverted_cdf")


def negative_gradient(y, F, loss, delta=None):
    """Pseudo-residual r = -dL/dF for every sample."""
    residual = y - F
    if loss == "squared_error":                        # L = 1/2 (y - F)^2
        return residual
    if loss == "absolute_error":                       # L = |y - F|
        return np.where(F > y, -1.0, 1.0)
    return np.where(np.abs(residual) <= delta, residual, np.where(residual > 0, delta, -delta))   # Huber


def leaf_value(y_leaf, F_leaf, r_leaf, loss, delta=None, median=None):
    """Line search in one leaf: the constant gamma that (approximately, for Huber) minimises the leaf's loss."""
    median = median or (lambda a: np.percentile(a, 50, method="inverted_cdf"))
    if loss == "squared_error":
        return np.mean(r_leaf)
    diff = y_leaf - F_leaf
    med = median(diff)
    if loss == "absolute_error":
        return med
    d = diff - med                                     # Huber: median + one robust correction step
    return med + np.mean(np.sign(d) * np.minimum(delta, np.abs(d)))


def regression_loss(y, F, loss, delta=None):
    """Average loss, as scikit-learn reports it for early stopping."""
    residual = y - F
    if loss == "squared_error":
        return np.mean(residual**2)
    if loss == "absolute_error":
        return np.mean(np.abs(residual))
    a = np.abs(residual)
    return np.mean(np.where(a <= delta, 0.5 * a**2, delta * (a - 0.5 * delta)))


# check each leaf line search against brute force on one leaf of 7 rows
y_leaf = np.array([3.0, 4.5, 5.0, 6.0, 7.5, 9.0, 30.0])        # one outlier
F_leaf = np.full(7, 5.0)
gammas = np.linspace(-5, 10, 150_001)                           # candidate shifts, step 0.0001
delta = 2.0
for loss in ["squared_error", "absolute_error", "huber"]:
    a = np.abs(y_leaf - (F_leaf + gammas[:, None]))             # |error| for every candidate (rows) and sample
    per_sample = {"squared_error": 0.5 * a**2, "absolute_error": a,
                  "huber": np.where(a <= delta, 0.5 * a**2, delta * (a - 0.5 * delta))}[loss]
    brute = gammas[int(np.argmin(per_sample.sum(axis=1)))]
    print(f"{loss:<15} leaf_value = {leaf_value(y_leaf, F_leaf, negative_gradient(y_leaf, F_leaf, loss, delta), loss, delta):+.4f}"
          f"   brute-force minimiser = {brute:+.4f}")

squared_error   leaf_value = +4.2857   brute-force minimiser = +4.2857
absolute_error  leaf_value = +1.0000   brute-force minimiser = +1.0000
huber           leaf_value = +1.1429   brute-force minimiser = +1.2500


> **Implementation hurdle — which median?** With an even number of values, "the median" can mean the average of the two
> middle values (`np.median`) or the lower of them. scikit-learn starts the model with `np.median` but sets absolute /
> Huber **leaf** values with a weighted percentile that returns the **lower** middle value. Using `np.median` in the
> leaves gives a slightly different (equally valid) model — demonstrated below with the boosting loop of section 3.

In [8]:
even = np.array([1.0, 2.0, 4.0, 10.0])
print("np.median:", np.median(even), "  lower-middle (scikit-learn's leaf median):", np.percentile(even, 50, method="inverted_cdf"))

np.median: 3.0   lower-middle (scikit-learn's leaf median): 2.0


> **How to read this**
> * Squared loss: the best constant shift is the mean residual (+4.2857, pulled up by the outlier 30). Absolute loss:
>   the median residual (+1.0), unmoved by the outlier. Both match brute force.
> * Huber: the one-step formula gives +1.1429 while the true minimiser is +1.25 — a deliberate approximation (one
>   step of an iterative method, Friedman 2001, Algorithm 4): close, robust to the outlier, but not exact.
> * Medians: 3.0 vs 2.0 for the same four numbers.

---
## 3. The boosting loop (regression)

README §3 and §4.5. For m = 1 … M:

1. pseudo-residuals r = −∂L/∂F at the current F (for Huber, first re-set δ);
2. fit a regression tree to r (on all rows, or on a random subset — section 6);
3. replace each leaf's value by the leaf's line search γⱼ (squared loss: nothing to do);
4. **F ← F + η·γ** — the learning rate η shrinks the step *after* γ was chosen.

The model is just a dict: the starting value, the list of trees, and η.

In [9]:
def subsample_rows(n, n_inbag, rng):
    """Pick exactly n_inbag of n rows WITHOUT replacement (selection sampling, scikit-learn's method). Section 6."""
    u = rng.uniform(size=n)
    mask, taken = np.zeros(n, dtype=bool), 0
    for i in range(n):
        if u[i] * (n - i) < n_inbag - taken:
            mask[i] = True
            taken += 1
    return np.nonzero(mask)[0]


def fit_gbm_regressor(X, y, loss="squared_error", n_estimators=100, learning_rate=0.1, max_depth=3,
                      subsample=1.0, alpha=0.9, n_iter_no_change=None, validation_fraction=0.1, tol=1e-4,
                      random_state=0, leaf_fn=leaf_value):
    """Gradient boosting for regression. Returns {'F0', 'trees', 'learning_rate'}."""
    X, y = round_to_float32(X), np.asarray(y, dtype=np.float64)
    if n_iter_no_change is not None:                  # section 7: hold out a validation split
        tr, va = train_test_split(np.arange(len(y)), random_state=random_state, test_size=validation_fraction)
        X_val, y_val, X, y = X[va], y[va], X[tr], y[tr]
    n, rng = len(y), np.random.RandomState(random_state)
    F0 = initial_prediction(y, loss)
    F, trees = np.full(n, F0), []
    if n_iter_no_change is not None:
        F_val, history = np.full(len(y_val), F0), np.full(n_iter_no_change, np.inf)
    for m in range(n_estimators):
        rows = subsample_rows(n, max(1, int(subsample * n)), rng) if subsample < 1.0 else np.arange(n)
        delta = huber_delta(y, F, alpha) if loss == "huber" else None
        r = negative_gradient(y, F, loss, delta)                                          # step 1
        tree = fit_regression_tree(X, r, rows, max_depth, rng.randint(0, RAND_R_MAX))     # step 2
        leaves = apply_tree(tree, X)
        if loss != "squared_error":                                                       # step 3
            for leaf in np.unique(leaves[rows]):
                in_leaf = rows[leaves[rows] == leaf]
                tree["value"][leaf] = leaf_fn(y[in_leaf], F[in_leaf], r[in_leaf], loss, delta)
        F = F + learning_rate * tree["value"][leaves]                                     # step 4
        trees.append(tree)
        if n_iter_no_change is not None:              # section 7: stop when validation stops improving
            F_val = F_val + learning_rate * predict_tree(tree, X_val)
            score = regression_loss(y_val, F_val, loss, delta)
            if np.any(score + tol < history):
                history[m % n_iter_no_change] = score
            else:
                break
    return {"F0": F0, "trees": trees, "learning_rate": learning_rate}


def predict_gbm_regressor(model, X):
    X = round_to_float32(X)
    F = np.full(len(X), model["F0"])
    for tree in model["trees"]:
        F = F + model["learning_rate"] * predict_tree(tree, X)
    return F


# README 5.1: six points, three stumps, eta = 0.5
x_toy = np.arange(1, 7, dtype=float).reshape(-1, 1)
y_toy = np.array([1, 2, 3, 6, 8, 10], dtype=float)
toy = fit_gbm_regressor(x_toy, y_toy, n_estimators=3, learning_rate=0.5, max_depth=1)
print("F0 =", toy["F0"])
for m, tree in enumerate(toy["trees"], start=1):
    F_m = predict_gbm_regressor({**toy, "trees": toy["trees"][:m]}, x_toy)
    print(f"round {m}: split x <= {tree['threshold'][0]}, leaves {tree['value'][1:]}, F = {F_m}, "
          f"MSE = {np.mean((y_toy - F_m) ** 2):.4f}")
assert np.allclose(predict_gbm_regressor(toy, x_toy), [2.65, 2.65, 2.65, 5.65, 7.525, 8.875])
print("matches README 5.1 ✔   prediction at x = 100:", predict_gbm_regressor(toy, np.array([[100.0]])))

# the which-median hurdle, now in a whole model (absolute loss, good-fit data, 100 rounds)
sk_abs = GradientBoostingRegressor(loss="absolute_error", random_state=0).fit(X_good, y_good)
for label, med in [("lower-middle median (scikit-learn's)", None), ("np.median in the leaves", np.median)]:
    model = fit_gbm_regressor(X_good, y_good, loss="absolute_error",
                              leaf_fn=lambda yl, Fl, rl, loss, d, med=med: leaf_value(yl, Fl, rl, loss, d, median=med))
    print(f"{label:<38}: max |difference| from scikit-learn = {np.abs(predict_gbm_regressor(model, X_good) - sk_abs.predict(X_good)).max():.2e}")

F0 = 5.0
round 1: split x <= 3.5, leaves [-3.  3.], F = [3.5 3.5 3.5 6.5 6.5 6.5], MSE = 3.9167
round 2: split x <= 4.5, leaves [-1.25  2.5 ], F = [2.875 2.875 2.875 5.875 7.75  7.75 ], MSE = 1.5729
round 3: split x <= 5.5, leaves [-0.45  2.25], F = [2.65  2.65  2.65  5.65  7.525 8.875], MSE = 0.8135
matches README 5.1 ✔   prediction at x = 100: [8.875]


lower-middle median (scikit-learn's)  : max |difference| from scikit-learn = 2.84e-14


np.median in the leaves               : max |difference| from scikit-learn = 1.22e+00


> **Implementation hurdle — shrink AFTER the line search, not inside it.** The old notes claimed that putting η inside
> the argmin "would not make any difference". It makes all the difference: if we choose γ to minimise
> Σ L(y, F + η·γ), the optimiser simply returns γ/η and the step η·γ is the full, unshrunk step — exactly η = 1.

In [10]:
X_tr, X_va, y_tr, y_va = train_test_split(X_good, y_good, test_size=0.25, random_state=0)


def leaf_value_eta_inside(y_leaf, F_leaf, r_leaf, loss, delta=None, eta=0.1):
    """WRONG on purpose: gamma = argmin sum L(y, F + eta * gamma) = (ordinary leaf value) / eta."""
    return leaf_value(y_leaf, F_leaf, r_leaf, loss, delta) / eta


variants = {"eta = 0.1, applied after the line search": fit_gbm_regressor(X_tr, y_tr, loss="absolute_error", n_estimators=200),
            "eta = 0.1 inside the argmin": fit_gbm_regressor(X_tr, y_tr, loss="absolute_error", n_estimators=200,
                                                             leaf_fn=leaf_value_eta_inside),
            "eta = 1 (no shrinkage)": fit_gbm_regressor(X_tr, y_tr, loss="absolute_error", n_estimators=200, learning_rate=1.0)}
preds_va = {k: predict_gbm_regressor(m, X_va) for k, m in variants.items()}
for k, p in preds_va.items():
    print(f"{k:<42} validation MAE {np.mean(np.abs(y_va - p)):.3f}")
print("'inside' == 'eta = 1' prediction for prediction:",
      np.allclose(preds_va["eta = 0.1 inside the argmin"], preds_va["eta = 1 (no shrinkage)"]))

sk_sq = GradientBoostingRegressor(random_state=0).fit(X_good, y_good)
own_sq = fit_gbm_regressor(X_good, y_good)
print("\nsquared loss, good-fit, 100 trees of depth 3: max |own - scikit-learn| =",
      np.abs(predict_gbm_regressor(own_sq, X_good) - sk_sq.predict(X_good)).max())

eta = 0.1, applied after the line search   validation MAE 1.046
eta = 0.1 inside the argmin                validation MAE 1.724
eta = 1 (no shrinkage)                     validation MAE 1.724
'inside' == 'eta = 1' prediction for prediction: True



squared loss, good-fit, 100 trees of depth 3: max |own - scikit-learn| = 7.105427357601002e-15


> **How to read this**
> * The hand example reproduces README §5.1: splits at 3.5, 4.5, 5.5; leaves −3/+3, −1.25/+2.5, −0.45/+2.25; final
>   F = 2.65, 2.65, 2.65, 5.65, 7.525, 8.875; and the flat value 8.875 far outside the data.
> * Medians in the leaves: with scikit-learn's lower-middle median the absolute-loss model matches scikit-learn to
>   3e-14; with `np.median` it differs by up to 1.22 (both are legitimate medians).
> * η inside the argmin: predictions identical to η = 1 — the shrinkage is completely undone — and validation MAE
>   **1.724** instead of **1.046** with real shrinkage. The squared-loss model matches scikit-learn to 7e-15.

---
## 4. Binary classification

README §4.4 and §5.2. Work in **log-odds** F; the probability is p = sigmoid(F) = 1/(1 + e^(−F)). With the log-loss
L = ln(1 + e^F) − yF:

* F₀ = ln(p̄ / (1 − p̄)) where p̄ is the share of class 1 (the log-odds, **not** p̄ itself);
* r = −∂L/∂F = y − p;
* the leaf value is one Newton step: γⱼ = Σ r / Σ p(1 − p) (∂²L/∂F² = p(1 − p)).

In [11]:
def sigmoid(F):
    """1 / (1 + e^-F) without overflow: use e^F / (1 + e^F) when F is negative."""
    F = np.asarray(F, dtype=np.float64)
    out = np.empty_like(F)
    pos = F >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-F[pos]))
    e = np.exp(F[~pos])
    out[~pos] = e / (1.0 + e)
    return out


def binary_negative_gradient(y, F):
    """r = y - sigmoid(F), in scikit-learn's overflow-safe form."""
    e = np.exp(-np.maximum(F, -37.0))
    return -np.where(F > -37.0, ((1 - y) - y * e) / (1 + e), np.exp(np.minimum(F, -37.0)) - y)


def initial_raw_scores(y, n_classes):
    """F0: log-odds of the class prior (binary), or symmetric log-priors that sum to 0 (K classes, section 5)."""
    eps = np.finfo(np.float64).eps
    prior = np.clip(np.bincount(y, minlength=n_classes) / len(y), eps, 1 - eps)
    if n_classes == 2:
        return np.array([np.log(prior[1] / (1 - prior[1]))])
    return np.log(prior / np.exp(np.mean(np.log(prior))))       # ln(p_k / geometric mean of p)


def safe_divide(numerator, denominator):
    """Newton step guard: a leaf whose probabilities are all ~0 or ~1 has nothing left to learn -> 0."""
    return 0.0 if abs(denominator) < 1e-150 else numerator / denominator


def newton_leaf_value(r_leaf, p_leaf, n_classes):
    """gamma = sum r / sum p(1-p)   (times (K-1)/K for K > 2 classes, scikit-learn's convention)."""
    factor = (n_classes - 1) / n_classes if n_classes > 2 else 1.0
    return safe_divide(np.mean(r_leaf) * factor, np.mean(p_leaf * (1 - p_leaf)))


# README 5.2: the placement data, with the leaf memberships from the screenshot
F = np.full(8, initial_raw_scores(y_place, 2)[0])
print(f"F0 = {F[0]:.6f} (= ln 5/3 = {np.log(5 / 3):.6f})")
for m, groups in enumerate([[[1, 2, 3], [0, 7], [4, 5, 6]], [[0, 1, 2, 3], [4], [5, 6, 7]]], start=1):
    p, r = sigmoid(F), binary_negative_gradient(y_place, F)
    gammas = [newton_leaf_value(r[g], p[g], 2) for g in groups]
    for g, gamma in zip(groups, gammas):
        F[g] += 1.0 * gamma                                    # learning rate 1, as in the lecture
    print(f"tree {m}: leaves (rows) {[[i + 1 for i in g] for g in groups]} -> gamma {np.round(gammas, 4)}")
F_query = initial_raw_scores(y_place, 2)[0] + (-1.25 / (2 * 0.625 * 0.375)) + gammas[2]
print(f"query (cgpa 7.2, iq 100): F = {F_query:.4f}, p = {sigmoid(np.array([F_query]))[0]:.4f}")

F0 = 0.510826 (= ln 5/3 = 0.510826)
tree 1: leaves (rows) [[2, 3, 4], [1, 8], [5, 6, 7]] -> gamma [ 1.6    -2.6667  0.1778]
tree 2: leaves (rows) [[1, 2, 3, 4], [5], [6, 7, 8]] -> gamma [ 0.5767 -2.9909  1.0497]
query (cgpa 7.2, iq 100): F = -1.1061, p = 0.2486


> **Implementation hurdle — numerical stability of sigmoid and log-loss.** For large |F| the textbook formulas overflow:
> e^1000 is `inf` in float64, and inf/inf is `nan`. The stable versions pick the form that only ever exponentiates a
> negative number. scikit-learn additionally uses a special branch below F = −37 (where sigmoid(F) < machine epsilon).

In [12]:
F_extreme = np.array([-1000.0, -40.0, 0.0, 40.0, 1000.0])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    naive_a = 1.0 / (1.0 + np.exp(-F_extreme))
    naive_b = np.exp(F_extreme) / (1.0 + np.exp(F_extreme))
    naive_loss = np.log(1.0 + np.exp(F_extreme))
for w in caught:
    print("warning caught:", w.category.__name__, "-", w.message)
print("naive 1/(1+e^-F)      :", naive_a)
print("naive e^F/(1+e^F)     :", naive_b)
print("stable sigmoid        :", sigmoid(F_extreme))
print("naive ln(1+e^F)       :", naive_loss)
print("stable logaddexp(0, F):", np.logaddexp(0, F_extreme))

warning caught: RuntimeWarning - overflow encountered in exp
warning caught: RuntimeWarning - overflow encountered in exp
warning caught: RuntimeWarning - overflow encountered in exp
warning caught: RuntimeWarning - invalid value encountered in divide
warning caught: RuntimeWarning - overflow encountered in exp
naive 1/(1+e^-F)      : [0.  0.  0.5 1.  1. ]
naive e^F/(1+e^F)     : [0.  0.  0.5 1.  nan]
stable sigmoid        : [0.  0.  0.5 1.  1. ]
naive ln(1+e^F)       : [ 0.      0.      0.6931 40.         inf]
stable logaddexp(0, F): [   0.        0.        0.6931   40.     1000.    ]


> **Implementation hurdle — the Newton denominator.** When every probability in a leaf is (numerically) 0 or 1, Σp(1−p)
> is 0 and the Newton step is 0/0. In float64, p = sigmoid(40) is *exactly* 1.0, so this really happens in long runs on
> separable data. scikit-learn returns 0 for such leaves (|denominator| < 1e-150): nothing is left to learn there.

In [13]:
p_saturated = sigmoid(np.array([40.0, 45.0, 50.0]))
r_saturated = 1.0 - p_saturated
print("p =", p_saturated, " exactly 1.0:", np.all(p_saturated == 1.0), " sum p(1-p) =", np.sum(p_saturated * (1 - p_saturated)))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    unguarded = np.mean(r_saturated) / np.mean(p_saturated * (1 - p_saturated))
print("unguarded Newton step:", unguarded, "| warning:", caught[0].category.__name__ if caught else None)
print("guarded Newton step  :", newton_leaf_value(r_saturated, p_saturated, 2))

p = [1. 1. 1.]  exactly 1.0: True  sum p(1-p) = 0.0
unguarded Newton step: nan | warning: RuntimeWarning
guarded Newton step  : 0.0


> **Implementation hurdle — the old tutorial's three bugs.** The previous notes copied a `CustomGradientBoostingClassifier`
> from a blog post. Its log-loss matched scikit-learn on the training data, which hid three bugs. We reproduce the
> class as it was (it uses scikit-learn's tree, which is fine for a demonstration) and trigger each bug.

In [14]:
class CustomGradientBoostingClassifier:          # copied from the old notes, bugs included
    def __init__(self, learning_rate, n_estimators, max_depth=1):
        self.learning_rate, self.n_estimators, self.max_depth, self.trees = learning_rate, n_estimators, max_depth, []

    def fit(self, X, y):
        F0 = np.log(y.mean() / (1 - y.mean()))
        self.F0 = np.full(len(y), F0)            # BUG 3: stored with the TRAINING length
        Fm = self.F0.copy()
        for _ in range(self.n_estimators):
            p = np.exp(Fm) / (1 + np.exp(Fm))
            r = y - p
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=0).fit(X, r)
            ids = tree.apply(x)                  # BUG 1: the GLOBAL x, not the argument X
            for j in np.unique(ids):
                fltr = ids == j
                gamma = r[fltr].sum() / (p[fltr] * (1 - p[fltr])).sum()
                Fm[fltr] += self.learning_rate * gamma
                tree.tree_.value[j, 0, 0] = gamma
            self.trees.append(tree)

    def predict_proba(self, X):
        Fm = self.F0                             # BUG 2: an alias, so += below changes self.F0 itself
        for i in range(self.n_estimators):
            Fm += self.learning_rate * self.trees[i].predict(X)
        return np.exp(Fm) / (1 + np.exp(Fm))


Xc, yc = make_circles(n_samples=100, factor=0.5, noise=0.15, random_state=0)
keep = np.sort(np.append(np.where(yc == 1)[0], np.where(yc == 0)[0][:-10]))
Xc, yc = Xc[keep], yc[keep]
x = Xc                                           # the global the buggy code silently relies on

old = CustomGradientBoostingClassifier(learning_rate=0.1, n_estimators=20)
old.fit(Xc, yc)
first, second = old.predict_proba(Xc), old.predict_proba(Xc)
print("bug 2: two calls of predict_proba on the same data differ by", np.abs(first - second).max())
try:
    old.predict_proba(Xc[:5])
except ValueError as e:
    print("bug 3: predicting 5 rows fails ->", str(e)[:80])
try:
    CustomGradientBoostingClassifier(learning_rate=0.1, n_estimators=20).fit(Xc[:60], yc[:60])
except IndexError as e:
    print("bug 1: fitting on a subset fails ->", str(e)[:80])


def fit_gbm_classifier(X, y, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0,
                       n_iter_no_change=None, validation_fraction=0.1, tol=1e-4, random_state=0):
    """Gradient boosting for binary (1 tree per round) and K-class (K trees per round) classification."""
    X = round_to_float32(X)
    classes, y = np.unique(y, return_inverse=True)              # columns follow the sorted class labels
    K = len(classes)
    n_trees = 1 if K == 2 else K
    if n_iter_no_change is not None:                            # section 7
        tr, va = train_test_split(np.arange(len(y)), random_state=random_state,
                                  test_size=validation_fraction, stratify=y)
        X_val, y_val, X, y = X[va], y[va], X[tr], y[tr]
    n, rng = len(y), np.random.RandomState(random_state)
    Y = np.eye(K)[y] if K > 2 else y[:, None].astype(np.float64)   # one-hot targets (one column if binary)
    F0 = initial_raw_scores(y, K)
    F, rounds = np.tile(F0, (n, 1)), []
    if n_iter_no_change is not None:
        F_val, history = np.tile(F0, (len(y_val), 1)), np.full(n_iter_no_change, np.inf)
    for m in range(n_estimators):
        rows = subsample_rows(n, max(1, int(subsample * n)), rng) if subsample < 1.0 else np.arange(n)
        R = binary_negative_gradient(Y, F) if K == 2 else Y - softmax(F)   # all residuals use the round-start F
        trees = []
        for k in range(n_trees):
            tree = fit_regression_tree(X, R[:, k], rows, max_depth, rng.randint(0, RAND_R_MAX))
            leaves = apply_tree(tree, X)
            for leaf in np.unique(leaves[rows]):
                in_leaf = rows[leaves[rows] == leaf]
                p = Y[in_leaf, k] - R[in_leaf, k]                       # p recovered from r = y - p
                tree["value"][leaf] = newton_leaf_value(R[in_leaf, k], p, K)
            F[:, k] += learning_rate * tree["value"][leaves]
            trees.append(tree)
        rounds.append(trees)
        if n_iter_no_change is not None:
            for k, tree in enumerate(trees):
                F_val[:, k] += learning_rate * predict_tree(tree, X_val)
            score = log_loss_from_scores(y_val, F_val)
            if np.any(score + tol < history):
                history[m % n_iter_no_change] = score
            else:
                break
    return {"F0": F0, "rounds": rounds, "learning_rate": learning_rate, "classes": classes}


def log_loss_from_scores(y, F):
    """Early-stopping score: binary deviance (2 x log-loss) or multinomial log-loss, computed from raw scores."""
    if F.shape[1] == 1:
        return 2 * np.mean(np.logaddexp(0, F[:, 0]) - y * F[:, 0])
    top = F.max(axis=1)
    return np.mean(np.log(np.exp(F - top[:, None]).sum(axis=1)) + top - F[np.arange(len(y)), y])


def raw_scores_gbm(model, X):
    X = round_to_float32(X)
    F = np.tile(model["F0"], (len(X), 1))
    for trees in model["rounds"]:
        for k, tree in enumerate(trees):
            F[:, k] += model["learning_rate"] * predict_tree(tree, X)
    return F


def predict_proba_gbm(model, X):
    F = raw_scores_gbm(model, X)
    if len(model["classes"]) == 2:
        p = sigmoid(F[:, 0])
        return np.column_stack([1 - p, p])
    return softmax(F)


def softmax(F):
    """exp(F_k) / sum_j exp(F_j), after subtracting each row's maximum (section 5)."""
    Z = np.exp(F - F.max(axis=1, keepdims=True))
    return Z / Z.sum(axis=1, keepdims=True)


fixed = fit_gbm_classifier(Xc, yc, n_estimators=20, learning_rate=0.1, max_depth=1)
p_a, p_b = predict_proba_gbm(fixed, Xc), predict_proba_gbm(fixed, Xc)
print("\nfixed version: two calls identical:", np.array_equal(p_a, p_b), "| 5 rows work:", predict_proba_gbm(fixed, Xc[:5]).shape)
print(f"log-loss (20 stumps, eta 0.1): own {log_loss(yc, p_a[:, 1]):.15f}   old notes printed 0.461943067988696")
sk_c = GradientBoostingClassifier(n_estimators=20, learning_rate=0.1, max_depth=1, random_state=0).fit(Xc, yc)
print("max |own - scikit-learn| probability:", np.abs(p_a - sk_c.predict_proba(Xc)).max())

bug 2: two calls of predict_proba on the same data differ by 0.17213464612536722
bug 3: predicting 5 rows fails -> operands could not be broadcast together with shapes (90,) (5,) (90,) 
bug 1: fitting on a subset fails -> boolean index did not match indexed array along axis 0; size of axis is 60 but s

fixed version: two calls identical: True | 5 rows work: (5, 2)
log-loss (20 stumps, eta 0.1): own 0.461943067988696   old notes printed 0.461943067988696
max |own - scikit-learn| probability: 2.220446049250313e-16


> **How to read this**
> * Placement data: F₀ = 0.510826; tree 1 leaves **1.6, −2.6667, 0.1778** (the screenshot's 1.6, −2.66, 0.18); tree 2
>   leaves **0.5767, −2.9909, 1.0497** (not the screenshot's 1.09, −2.94, 1.56 — see README §5.2); query point F = −1.1061,
>   p = 0.2486.
> * Stability: the naive formulas print overflow warnings, return `nan` for e^F/(1+e^F) at F = 1000 and `inf` for the
>   loss; the stable versions return 0, 1 and the correct loss 1000.
> * Newton guard: at F = 40 the probability is exactly 1.0, Σp(1−p) = 0, the unguarded step is `nan` (with a warning);
>   the guarded step is 0.
> * The three old bugs are real: repeated `predict_proba` calls drift (bug 2), 5-row prediction crashes (bug 3),
>   fitting on 60 rows crashes because `tree.apply(x)` reads the 90-row global (bug 1). The functional version has
>   none of them and still reproduces the old notes' log-loss **0.461943067988696** and scikit-learn's probabilities.

---
## 5. Multi-class classification: softmax and K trees per round

README §4.6. Each sample has K scores F₁…F_K; probabilities are the softmax p_k = e^{F_k} / Σ_j e^{F_j}. The residual
for class k is y_k − p_k (y_k = 1 if the sample is in class k, else 0). Each round fits **one tree per class** to its
residuals (all computed from the scores at the start of the round), and each leaf gets the diagonal Newton step
(K−1)/K · Σr / Σp(1−p) — "diagonal" because it ignores how changing class k's score also moves the other classes'
probabilities. The start F₀ is ln p_k minus the average of the ln p_k, so the scores sum to 0.

In [15]:
F_big = np.array([[1000.0, 999.0, 0.0], [-5.0, 0.0, 5.0]])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    naive = np.exp(F_big) / np.exp(F_big).sum(axis=1, keepdims=True)
print("naive softmax :", naive.round(4).tolist(), "| warnings:", sorted({w.category.__name__ for w in caught}))
print("stable softmax:", softmax(F_big).round(4).tolist())
print("adding 100 to every score changes nothing:", np.allclose(softmax(F_big[1:] + 100), softmax(F_big[1:])))

y3 = np.digitize(y_good, np.quantile(y_good, [1 / 3, 2 / 3]))             # 3 classes: low / middle / high y
F0_3 = initial_raw_scores(y3, 3)
print("\nclass shares:", np.bincount(y3) / len(y3), " F0 =", F0_3, " sum =", F0_3.sum())

own3 = fit_gbm_classifier(X_good, y3, n_estimators=100)
sk3 = GradientBoostingClassifier(n_estimators=100, random_state=0).fit(X_good, y3)
print("rounds x trees per round: own", (len(own3["rounds"]), len(own3["rounds"][0])), "scikit-learn", sk3.estimators_.shape)
print("max |own - scikit-learn| probability:", np.abs(predict_proba_gbm(own3, X_good) - sk3.predict_proba(X_good)).max())

naive softmax : [[nan, nan, 0.0], [0.0, 0.0067, 0.9933]] | warnings: ['RuntimeWarning']
stable softmax: [[0.7311, 0.2689, 0.0], [0.0, 0.0067, 0.9933]]
adding 100 to every score changes nothing: True

class shares: [0.3335 0.333  0.3335]  F0 = [ 0.0005 -0.001   0.0005]  sum = 3.5258254649228604e-16


rounds x trees per round: own (100, 3) scikit-learn (100, 3)
max |own - scikit-learn| probability: 3.3306690738754696e-16


> **Implementation hurdle — class-order alignment.** With string labels the columns of the probability matrix follow
> the **sorted** labels; our `np.unique` and scikit-learn's `classes_` agree, so column k means the same class in both.

In [16]:
labels = np.array(["mid", "high", "low"])[y3]                             # shuffle the meaning: strings, not 0/1/2
own_s = fit_gbm_classifier(X_good, labels, n_estimators=20)
sk_s = GradientBoostingClassifier(n_estimators=20, random_state=0).fit(X_good, labels)
print("own classes:", own_s["classes"], " scikit-learn classes_:", sk_s.classes_)
print("max |difference| column by column:", np.abs(predict_proba_gbm(own_s, X_good) - sk_s.predict_proba(X_good)).max())

own classes: ['high' 'low' 'mid']  scikit-learn classes_: ['high' 'low' 'mid']
max |difference| column by column: 2.220446049250313e-16


> **How to read this**
> * The naive softmax turns e^1000 into `inf` and returns `nan`; subtracting the row maximum first (the log-sum-exp
>   trick) is exact and safe, and adding a constant to all scores changes nothing — the over-parametrisation.
> * F₀ sums to 0 (3.5e-16, rounding), 100 rounds × 3 trees, and the probabilities match scikit-learn to 3e-16.
> * With string labels the column order is "high, low, mid" (sorted) in both implementations.

---
## 6. Subsampling: a new random half every round, without replacement

README §6.2. scikit-learn draws the in-bag rows with **selection sampling**: walk through the rows once and take row i
with probability (still needed) / (still left). That always yields exactly ⌊subsample·n⌋ distinct rows. Because
our generator consumes the random numbers in the same order as scikit-learn's (one `uniform` per row each round, one
integer per tree for the feature order), even subsampled models match exactly.

In [17]:
masks = []
GradientBoostingRegressor(n_estimators=3, subsample=0.5, random_state=0).fit(
    X_good, y_good, monitor=lambda i, est, local_vars: masks.append(local_vars["sample_mask"].copy()) or False)
rng_check = np.random.RandomState(0)
for m in range(3):
    rows_m = subsample_rows(len(y_good), 1000, rng_check)
    rng_check.randint(0, RAND_R_MAX)                                     # the tree's seed draw
    print(f"round {m + 1}: {len(rows_m)} rows, {len(np.unique(rows_m))} distinct, same rows as scikit-learn: "
          f"{np.array_equal(np.nonzero(masks[m])[0], rows_m)}")

X_raw, y_raw = X_tr, y_tr                                               # the 1500-row split from section 3
own_sub = fit_gbm_regressor(X_raw, y_raw, subsample=0.5)
sk_sub = GradientBoostingRegressor(subsample=0.5, random_state=0).fit(X_raw, y_raw)
print("subsample = 0.5, 100 trees, 1500 rows: max |own - scikit-learn| =",
      np.abs(predict_gbm_regressor(own_sub, X_va) - sk_sub.predict(X_va)).max())

# the same on all 2000 rows of the CSV: tied x values (the CSV keeps 6 digits) meet an exact tie in round 95
own_all = fit_gbm_regressor(X_good, y_good, subsample=0.5)
sk_all = GradientBoostingRegressor(subsample=0.5, random_state=0).fit(X_good, y_good)
first_diff = next(m for m, (t_own, t_sk) in enumerate(zip(own_all["trees"], sk_all.estimators_[:, 0]))
                  if not np.array_equal(t_own["feature"], np.where(t_sk.tree_.feature < 0, -1, t_sk.tree_.feature)))
print("duplicated values per feature in the CSV (after float32 rounding):",
      [len(y_good) - len(np.unique(round_to_float32(X_good)[:, j])) for j in range(10)])
print(f"2000 rows: first {first_diff} trees identical; tree {first_diff + 1} splits a 15-row node on another feature")
print(f"  training RMSE own {np.sqrt(mean_squared_error(y_good, predict_gbm_regressor(own_all, X_good))):.4f} vs scikit-learn "
      f"{np.sqrt(mean_squared_error(y_good, sk_all.predict(X_good))):.4f}")

round 1: 1000 rows, 1000 distinct, same rows as scikit-learn: True
round 2: 1000 rows, 1000 distinct, same rows as scikit-learn: True
round 3: 1000 rows, 1000 distinct, same rows as scikit-learn: True


subsample = 0.5, 100 trees, 1500 rows: max |own - scikit-learn| = 3.552713678800501e-15


duplicated values per feature in the CSV (after float32 rounding): [0, 1, 1, 2, 3, 0, 4, 5, 2, 0]
2000 rows: first 94 trees identical; tree 95 splits a 15-row node on another feature
  training RMSE own 1.0536 vs scikit-learn 1.0533


---
## 7. Early stopping on a validation split

README §6.2. With `n_iter_no_change = k`, hold out `validation_fraction` of the training rows (stratified for
classification), track the validation loss after every round, and stop when it has not beaten any of the last k
recorded values by more than `tol`. The tree of the stopping round is kept.

In [18]:
for name, own_fit, sk_model, X_, y_ in [
        ("regression, squared", lambda: fit_gbm_regressor(X_good, y_good, n_estimators=2000, n_iter_no_change=10),
         GradientBoostingRegressor(n_estimators=2000, n_iter_no_change=10, random_state=0), X_good, y_good),
        ("3-class", lambda: fit_gbm_classifier(X_good, y3, n_estimators=2000, n_iter_no_change=10),
         GradientBoostingClassifier(n_estimators=2000, n_iter_no_change=10, random_state=0), X_good, y3)]:
    own = own_fit()
    sk_model.fit(X_, y_)
    n_own = len(own["trees"]) if "trees" in own else len(own["rounds"])
    print(f"{name:<20}: own stopped after {n_own} rounds, scikit-learn n_estimators_ = {sk_model.n_estimators_}")

regression, squared : own stopped after 168 rounds, scikit-learn n_estimators_ = 168


3-class             : own stopped after 220 rounds, scikit-learn n_estimators_ = 220


> **How to read this**
> * Each round uses exactly 1000 distinct rows (half of 2000), a fresh set each round, identical to scikit-learn's.
>   On the 1500-row training split the subsampled model therefore matches scikit-learn exactly (not just "in
>   distribution").
> * **Implementation hurdle — tied x values.** On all 2000 rows the two models agree for the first 94 trees; tree 95
>   meets an exact tie (two features cut a 15-row node into the same two groups) and it is broken differently. The
>   cause: the CSV stores 6 digits, so a few rows share the same x value; scikit-learn sorts with an *introsort*, which
>   orders tied rows differently from our stable sort, which changes the last bit of a sum (hurdle (c) in §1.2). After
>   that the models are different but equally good (same training RMSE to 3 decimals). Copying scikit-learn's sort
>   in Python would be far too slow, so we stop here.
> * Early stopping stops at the same round as scikit-learn for both regression and classification.

---
## 8. A pointer: histogram binning (what `HistGradientBoosting*` and XGBoost's `hist` do)

The exact split search sorts every feature in every node. **Histogram** methods bin each feature once into at most
255 buckets (by quantiles); a node then only needs the sum of residuals per bucket (`np.bincount`) and a cumulative
sum over ≤ 255 buckets. The threshold can only sit on a bucket edge, which barely matters. Full treatment in
[`../06-xgboost/`](../../06-xgboost/README.md).

In [19]:
def bin_feature(x, n_bins=255):
    """Bucket edges at quantiles; returns the bucket index of every value and the edges."""
    edges = np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)[1:-1]))
    return np.searchsorted(edges, x, side="left"), edges


def best_threshold_histogram(bins, edges, r):
    """Best cut using only per-bucket sums: O(n + n_bins) per node instead of a sort."""
    counts = np.bincount(bins, minlength=len(edges) + 1)
    sums = np.bincount(bins, weights=r, minlength=len(edges) + 1)
    n_left, s_left = np.cumsum(counts)[:-1], np.cumsum(sums)[:-1]
    ok = (n_left > 0) & (n_left < len(r))
    score = np.where(ok, s_left**2 / np.maximum(n_left, 1) + (r.sum() - s_left) ** 2 / np.maximum(len(r) - n_left, 1), -np.inf)
    return edges[int(np.argmax(score))]


rng = np.random.default_rng(3)
x_h = rng.uniform(0, 1, 500_000)
r_h = np.sin(6 * x_h) + rng.normal(0, 0.3, len(x_h))
t0 = time.perf_counter(); o = np.argsort(x_h); pos, sc = score_splits(x_h[o], r_h[o]); t_exact = time.perf_counter() - t0
thr_exact = (x_h[o][pos[np.argmax(sc)] - 1] + x_h[o][pos[np.argmax(sc)]]) / 2
bins_h, edges_h = bin_feature(x_h)
t0 = time.perf_counter(); thr_hist = best_threshold_histogram(bins_h, edges_h, r_h); t_hist = time.perf_counter() - t0
print(f"exact (sort + cumsum): threshold {thr_exact:.4f} in {t_exact * 1e3:.0f} ms")
print(f"histogram (255 bins) : threshold {thr_hist:.4f} in {t_hist * 1e3:.0f} ms  (binning itself is done once per feature)")

exact (sort + cumsum): threshold 0.5285 in 38 ms
histogram (255 bins) : threshold 0.5266 in 1 ms  (binning itself is done once per feature)


> **How to read this**
> * On half a million rows the histogram search finds practically the same threshold (it can only choose a bucket
>   edge) many times faster per node, because the expensive sort happens once, before boosting starts.
> * This is the main idea behind `HistGradientBoostingRegressor` (Lab 18 of the library notebook) and XGBoost/LightGBM.

---
## 9. Comparison with scikit-learn on both CSVs

Same hyper-parameters (100 trees, η = 0.1, depth 3, `random_state=0`) on a train/test split of each CSV. Good fit:
random 75/25 split. Bad fit: the first 70 % of months for training, the last 30 % for testing (as in Lab 12). The
classification targets are derived from y: **binary** = "y above the training median", **3-class** = training-tertile
of y. Agreement = max |difference| of test predictions (regression) or test probabilities (classification).

In [20]:
def split_csv(name):
    if name == "good_fit":
        return train_test_split(X_good, y_good, test_size=0.25, random_state=0)
    cut = int(0.7 * len(y_bad))
    return X_bad[:cut], X_bad[cut:], y_bad[:cut], y_bad[cut:]


def timed(fn):
    t0 = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t0


rows_table = []
for data in ["good_fit", "bad_fit"]:
    Xa, Xb, ya, yb = split_csv(data)
    for loss in ["squared_error", "absolute_error", "huber"]:
        own, t_own = timed(lambda: fit_gbm_regressor(Xa, ya, loss=loss))
        sk, t_sk = timed(lambda: GradientBoostingRegressor(loss=loss, random_state=0).fit(Xa, ya))
        po, ps = predict_gbm_regressor(own, Xb), sk.predict(Xb)
        rows_table.append((data, f"regression, {loss}", "RMSE", np.sqrt(mean_squared_error(yb, po)),
                           np.sqrt(mean_squared_error(yb, ps)), np.abs(po - ps).max(), t_own, t_sk))
    median, tertiles = np.median(ya), np.quantile(ya, [1 / 3, 2 / 3])
    for task, ca, cb in [("binary (y > median)", (ya > median).astype(int), (yb > median).astype(int)),
                         ("3-class (tertiles)", np.digitize(ya, tertiles), np.digitize(yb, tertiles))]:
        own, t_own = timed(lambda: fit_gbm_classifier(Xa, ca))
        sk, t_sk = timed(lambda: GradientBoostingClassifier(random_state=0).fit(Xa, ca))
        po, ps = predict_proba_gbm(own, Xb), sk.predict_proba(Xb)
        rows_table.append((data, task, "log-loss", log_loss(cb, po, labels=np.arange(po.shape[1])),
                           log_loss(cb, ps, labels=np.arange(ps.shape[1])), np.abs(po - ps).max(), t_own, t_sk))
    Xa_, Xb_, ya_, yb_ = split_csv(data)
    own, t_own = timed(lambda: fit_gbm_regressor(Xa_, ya_, subsample=0.5))
    sk, t_sk = timed(lambda: GradientBoostingRegressor(subsample=0.5, random_state=0).fit(Xa_, ya_))
    po, ps = predict_gbm_regressor(own, Xb_), sk.predict(Xb_)
    rows_table.append((data, "regression, subsample 0.5", "RMSE", np.sqrt(mean_squared_error(yb_, po)),
                       np.sqrt(mean_squared_error(yb_, ps)), np.abs(po - ps).max(), t_own, t_sk))

Xc_a, Xc_b, yc_a, yc_b = train_test_split(*make_circles(n_samples=600, factor=0.5, noise=0.15, random_state=0),
                                          test_size=0.25, random_state=0)
own, t_own = timed(lambda: fit_gbm_classifier(Xc_a, yc_a))
sk, t_sk = timed(lambda: GradientBoostingClassifier(random_state=0).fit(Xc_a, yc_a))
po, ps = predict_proba_gbm(own, Xc_b), sk.predict_proba(Xc_b)
rows_table.append(("circles", "binary", "log-loss", log_loss(yc_b, po), log_loss(yc_b, ps), np.abs(po - ps).max(), t_own, t_sk))

print(f"{'data':<9}{'task':<28}{'metric':<9}{'own':>9}{'sklearn':>9}{'max |diff|':>12}{'own s':>7}{'sk s':>7}")
for data, task, metric, m_own, m_sk, diff, t_own, t_sk in rows_table:
    print(f"{data:<9}{task:<28}{metric:<9}{m_own:>9.4f}{m_sk:>9.4f}{diff:>12.1e}{t_own:>7.2f}{t_sk:>7.2f}")
print("\nlargest disagreement anywhere:", max(r[5] for r in rows_table))

data     task                        metric         own  sklearn  max |diff|  own s   sk s
good_fit regression, squared_error   RMSE        1.4782   1.4782     1.1e-14   0.36   0.20
good_fit regression, absolute_error  RMSE        1.5327   1.5327     2.1e-14   0.37   0.23
good_fit regression, huber           RMSE        1.4782   1.4782     7.1e-15   0.38   0.26
good_fit binary (y > median)         log-loss    0.2463   0.2463     3.3e-16   0.37   0.20
good_fit 3-class (tertiles)          log-loss    0.4861   0.4861     2.2e-16   1.11   0.60
good_fit regression, subsample 0.5   RMSE        1.3760   1.3760     3.6e-15   0.24   0.10
bad_fit  regression, squared_error   RMSE        9.4249   9.4249     1.4e-14   0.05   0.02
bad_fit  regression, absolute_error  RMSE        9.2272   9.2272     7.1e-14   0.05   0.05
bad_fit  regression, huber           RMSE        9.3576   9.3576     0.0e+00   0.06   0.07
bad_fit  binary (y > median)         log-loss    0.0019   0.0019     0.0e+00   0.05   0.02

> **How to read this**
> * Every row agrees to **floating-point precision** (largest difference printed above, of order 1e-14 for
>   regression targets of size ~10–60 and 1e-16 for probabilities) and the metrics are identical to 4 decimals —
>   including absolute and Huber loss, 3 classes, and `subsample=0.5`.
> * The bad-fit RMSEs (≈ 9–10) are large for both implementations: that is the failure from README §8.1, not a bug.
> * Fit time: the NumPy version is a little slower (Python loops over nodes, rows during partitioning, and features),
>   but in the same range as scikit-learn's Cython for these data sizes.

---
# Summary — what each section showed

| Section | Claim from the README | Verdict |
|---|---|---|
| 1.1 | Best split = max S_L²/n_L + S_R²/n_R; Friedman's criterion is the same; sort + cumsum is fast | ✔ identical cuts, hundreds of × faster |
| 1.2 | Our tree = scikit-learn's tree (float32, feature order, ties) | ✔ max diff 0 on 100,000 new rows |
| 2 | F₀ and leaf line searches: mean / median / Huber step | ✔ brute force agrees (Huber: one-step approximation) |
| 3 | Hand example §5.1; η applied after the line search | ✔ same numbers; η inside the argmin = no shrinkage |
| 4 | Binary: log-odds start, y − p, Newton leaf, stability, guard; old bugs fixed | ✔ |
| 5 | K classes: softmax, K trees, (K−1)/K, sum-zero start | ✔ matches to ~1e-16 |
| 6 | `subsample`: exactly ⌊0.5n⌋ distinct rows, new each round | ✔ same rows as scikit-learn |
| 7 | Early stopping on a validation split | ✔ same stopping round |
| 8 | Histogram binning ≈ same split, much cheaper | ✔ (pointer to 06-xgboost) |
| 9 | Own GBM = scikit-learn on both CSVs (all losses, binary, 3-class, subsample, circles) | ✔ to float precision |